#### This notebook contains code to validate the NASA AQcGAN model. 
It reads the NASA AQcGAN output *npz files and plots the errors wrt to the GEOS-CF output (truth). 
It assumes you have run the model in validation mode and expects both the prediction files, test_ens_pred_stats_daysX_level72.npz, and the truth files test_ens_stats_daysX_level72.npz.

In [ ]:
# Load modules
# On discover all modules are installed in the python/GEOSpyD/24.11.3-0/3.13 module
import xarray as xr
import os
import glob
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pickle

In [ ]:
# Path to the experiment directory
exp_path = "/discover/nobackup/projects/gmao/aist-cf/vshah5/GEOSAQcGAN/runs/v2.1.1_test/"

In [ ]:
# Read metadata (for lat,lon )
with open(f"{exp_path}/data/meta.pkl", "rb") as f:
    meta_data = pickle.load(f)

In [ ]:
outvars = ["CO","NO","NO2","O3"]
lead_times = [6, 12, 24, 48] # hours (max_n_passes = 2)

In [ ]:
# ds_concat is a dictionary with the predicted and truth arrays of ensemble means
# for the specified forecast lead times
ds_concat = {}

for lt in lead_times:
    n_pass = np.ceil(lt / 24).astype(int)
    pred_file = f"{exp_path}/exp/test_ens_pred_stats_days{n_pass}_level72.npz"     
    pred_data = np.load(pred_file)
    pred_mean = pred_data["ens_mean_pred_test"]

    truth_file = f"{exp_path}/exp/test_ens_stats_days{n_pass}_level72.npz"     
    truth_data = np.load(truth_file)
    truth_mean = truth_data["ens_mean_test"]
    
    ts = lt // ( 3 * n_pass ) - 1
    
    thisds = xr.Dataset(
                data_vars={
                    **{f"{var_name}_pred": (['time','lat','lon'], pred_mean[:,i,ts,:,:]) 
                    for i, var_name in enumerate(outvars)
                    },
                    **{f"{var_name}_truth": (['time','lat','lon'], truth_mean[:,i,ts,:,:]) 
                    for i, var_name in enumerate(outvars)
                    },
                    },
                coords= {
                    'lat': meta_data['lat'],
                    'lon': meta_data['lon'],
                    }
                )
    ds_concat[lt] = thisds

### Calculate error stats

In [ ]:
mb_results = {}
rmse_results = {}
acc_results = {}
truth_results = {}

for lead_time in lead_times:
    
    mb_results[lead_time] = {}
    rmse_results[lead_time] = {}
    acc_results[lead_time] = {}
    truth_results[lead_time] = {}

    for var in ds_concat[lead_time].data_vars:
        if var.endswith('_pred'):
            truth_var = var.replace('_pred', '_truth')
            if truth_var in ds_concat[lead_time]:
                pred = ds_concat[lead_time][var]
                
                # Set negative values in the predictions to 0
                pred = xr.where(pred > 0, pred, 0.0)
                
                truth = ds_concat[lead_time][truth_var]
                
                # truth                
                truth_results[lead_time][var] = truth.mean('time')
                
                # mean bias
                mb = ( pred - truth ).mean(['time'])
                mb_results[lead_time][var] = mb
                
                # RMSE
                mse = ((pred - truth)**2).mean(['time'])
                rmse = np.sqrt(mse)
                rmse_results[lead_time][var] = rmse

                # ACC for 24h mean values
                # Calculate 24h mean values to remove the effect of diurnal variation
                pred_24h_mean = pred.coarsen(time=8, boundary='trim').mean()
                truth_24h_mean = truth.coarsen(time=8, boundary='trim').mean()

                # Calculate anomalies
                pred_anomaly = (pred_24h_mean - pred_24h_mean.mean('time'))
                truth_anomaly = (truth_24h_mean - truth_24h_mean.mean('time'))

                # Calculate anomaly correlation coefficient
                acc = [ np.corrcoef(np.ravel(pred_anomaly.sel(time=t).values), 
                                    np.ravel(truth_anomaly.sel(time=t).values))[0, 1] 
                                    for t in pred_anomaly.time.values ]
                acc_results[lead_time][var] = np.mean(acc) # Mean for the month

ds_errstats = xr.Dataset()

for lead_time in lead_times:
    for var, truth in truth_results[lead_time].items():
        ds_errstats[f'{var}_truth_{lead_time}h'] = truth
    for var, mb in mb_results[lead_time].items():
        ds_errstats[f'{var}_mb_{lead_time}h'] = mb
    for var, rmse in rmse_results[lead_time].items():
        ds_errstats[f'{var}_rmse_{lead_time}h'] = rmse
    for var, acc in acc_results[lead_time].items():
        ds_errstats[f'{var}_acc_{lead_time}h'] = acc


## Relative RMSE

In [ ]:
from matplotlib import colors
norm = [0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6]
var_title = [r"O$_3$","CO",r"NO$_2$"]
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(16, 12), 
                        subplot_kw={'projection': ccrs.Robinson()})
# Plot each variable
for i, lead_time in enumerate(lead_times):
    for j, var_name in enumerate(["O3","CO","NO2"]):
        pred_var = [var for var in ds_errstats.variables if var_name in var and "rmse" in var and str(lead_time)+'h' in var]
        truth_var = [var for var in ds_errstats.variables if var_name in var]
        ax = axes[i,j]
        data = ds_errstats[pred_var[0]]/ds_errstats[truth_var[0]]
        if var_name == "NO2":
            # Ignore areas where NO2 < 1 ppbv
            data = data.where(ds_errstats[truth_var[0]] > 1e-9, np.nan)
        # Basic map setup
        ax.set_global()
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        # Plot data
        im = ax.pcolormesh(data.lon, data.lat, data.values, 
                        transform=ccrs.PlateCarree(),
                        norm=colors.BoundaryNorm(norm, 256),
                        cmap='RdPu')
        
        ax.set_title(pred_var[0], fontsize=12)

# Add shared colorbar
cbar_ax = fig.add_axes([0.94, 0.25, 0.03, 0.5])  # [left, bottom, width, height]
cbar = plt.colorbar(im, cax=cbar_ax, extend='max')
cbar.ax.tick_params(labelsize=24)
cbar.set_ticks(norm)
cbar.set_ticklabels([str(v) for v in norm])

# Adjust layout
plt.subplots_adjust(left=0.05, right=0.9, top=0.95, bottom=0.05, 
                    wspace=0.1, hspace=0.1)
plt.show()

## Anomaly correlation coefficient

In [ ]:

species = ['CO', 'NO2', 'O3']

for sp in species:
    acc_values = [ds_errstats[f"{sp}_pred_acc_{lt}h"].values for lt in lead_times]
    plt.plot(lead_times, acc_values, marker='o', label=sp)

plt.xlabel('Forecast lead time (hours)')
plt.ylabel('ACC')
plt.title('Anomaly correlation coefficient')
plt.legend()
plt.grid(True)
plt.show()

## Cities timeseries

In [ ]:
import requests
import pandas as pd
from geopy.geocoders import Nominatim

def get_city_coordinates(city_name):
    geolocator = Nominatim(user_agent="GEOSAQcGAN_app")
    location = geolocator.geocode(city_name)
    if location:
        return location.latitude, location.longitude
    return None, None

cities = [
    "Tokyo", "Delhi", "Shanghai", "São Paulo", "Mexico City",
    "Cairo", "Mumbai", "Beijing", "Dhaka",
    "New York", "Karachi", "Buenos Aires", "Chongqing", "Istanbul",
    "Kolkata", "Manila", "Lagos", "Rio de Janeiro", "London", "Johannesburg"
]

data = []
for city in cities:
    lat, lon = get_city_coordinates(city)
    data.append({'City': city, 'Latitude': lat, 'Longitude': lon})

df = pd.DataFrame(data)

In [ ]:
# Get the 20 largest cities
top_20_cities = df

# List of pollutants and lead times
pollutants = ['O3', 'CO', 'NO2']
lead_times = [6, 12, 24, 48]

# Create a figure with subplots for each city and pollutant
fig, axes = plt.subplots(len(top_20_cities), len(pollutants), figsize=(15*len(pollutants), 5*len(top_20_cities)), sharex=True)
#fig.suptitle('Time Series for Top 20 Cities', fontsize=16)

alpha_values = {6: 1.0, 12: 0.9, 24: 0.8, 48: 0.6, 96: 0.4}

for row, (_, city) in enumerate(top_20_cities.iterrows()):
    city_name = city['City']
    lat, lon = city['Latitude'], city['Longitude']
    for col, pollutant in enumerate(pollutants):
        ax = axes[row, col]
        
        # Plot truth and forecast data for each lead time
        for lead_time in lead_times:
            # Plot truth data
            plot_data = ds_concat[lead_time].sel(lat=lat, lon=lon, method='nearest')

            ax.plot(plot_data['time']+(lead_time-lead_times[0])//3, plot_data[f"{pollutant}_truth"],
                    label='Truth' if lead_time == lead_times[0] else None, color='k', alpha=1, lw=1)

            ax.plot(plot_data['time']+(lead_time-lead_times[0])//3, plot_data[f"{pollutant}_pred"],
                    label=f'{lead_time}h Forecast', color='rosybrown', alpha=alpha_values[lead_time], lw=3)

        ax.set_ylabel(f'{pollutant} Concentration', fontsize=24)
        if row == 0:
            ax.set_title(f'{pollutant} Time Series', fontsize=24)
            if col == len(pollutants) - 1:
                ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=24)
        if row == len(top_20_cities) - 1:
            ax.set_xlabel('Time', fontsize=14)
        ax.set_title(f'{city_name} - {pollutant}', fontsize=36)
        ax.tick_params(axis='both', which='major', labelsize=24)

plt.tight_layout()
plt.show()